In [ ]:
#!pip3 install -U torch datasets transformers==4.35.2 bitsandbytes peft==0.5.0 accelerate

In [5]:
!pip freeze
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

accelerate==0.24.1
aiohappyeyeballs==2.6.1
aiohttp==3.12.13
aiosignal==1.3.2
alembic==1.16.1
altair==5.5.0
annotated-types==0.7.0
anyio==4.7.0
appdirs==1.4.4
argon2-cffi==25.1.0
argon2-cffi-bindings==21.2.0
arrow==1.3.0
asttokens==2.4.1
async-lru==2.0.5
attrs==23.2.0
Babel==2.10.3
bcc==0.29.1
beautifulsoup4==4.12.3
bitsandbytes==0.46.0
bleach==6.2.0
blessed==1.21.0
blinker==1.9.0
blobfile==3.0.0
Brlapi==0.8.5
Brotli==1.1.0
cachetools==5.5.2
certifi==2023.11.17
cffi==1.17.1
chardet==5.2.0
charset-normalizer==3.4.2
click==8.1.6
cloud-init==25.1.4
cloudpickle==3.1.1
colorama==0.4.6
comm==0.2.2
command-not-found==0.3
configobj==5.0.8
contourpy==1.3.2
cryptography==41.0.7
cssselect==1.2.0
cupshelpers==1.0
cycler==0.12.1
databricks-sdk==0.55.0
datasets==3.6.0
dbus-python==1.3.2
debugpy==1.8.14
decorator==5.1.1
defer==1.0.6
defusedxml==0.7.1
Deprecated==1.2.18
dill==0.3.8
distro==1.9.0
distro-info==1.7+build1
docker==7.1.0
executing==2.0.1
fastapi==0.115.12
fastjsonschema==2.21.1
filelock==3.

In [2]:
import torch

from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, BitsAndBytesConfig

from peft import LoraConfig, get_peft_model, TaskType


/home/dk/.local/lib/python3.12/site-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/home/dk/.local/lib/python3.12/site-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


In [3]:
model_name = 'TinyLLama/TinyLlama-1.1B-Chat-v1.0'

bnb_config = BitsAndBytesConfig(
    load_in_4bit = True,
    bnb_4bit_quant_type = 'nf4',
    bnb_4bit_compute_dtype = torch.bfloat16
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map = 'auto',
    trust_remote_code = True
)

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

/home/dk/.local/lib/python3.12/site-packages/huggingface_hub/file_download.py:795: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [4]:
lora_config = LoraConfig(
    r = 8,
    lora_alpha = 16,
    target_modules = ['q_proj', 'v_proj'],
    lora_dropout = 0.05,
    bias = 'none',
    task_type = TaskType.CAUSAL_LM
)

model = get_peft_model(model, lora_config)

In [ ]:
data = load_dataset('json', data_files='kubernetes.jsonl')['train']

In [ ]:

def tokenize(batch):
    texts = [
        f"### Instruction:\n{inst}\n### Response:\n{out}"
        for inst, out in zip(batch['instruction'], batch['response'])
    ]

    tokens = tokenizer(
        texts,
        padding = 'max_length',
        truncation = True,
        max_length = 256,
        return_tensors = 'pt'
    )

    tokens['labels'] = tokens['input_ids'].clone()

    return tokens


In [ ]:
tokenized_data = data.map(tokenize, batched=True, remove_columns=data.column_names)

In [ ]:

training_args = TrainingArguments(
    output_dir = './tinyllama-lora-tuned-kubermetes',
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 2,
    learning_rate = 1e-3,
    num_train_epochs = 50,
    fp16 = True,
    logging_steps = 20,
    save_strategy = 'epoch',
    report_to = 'none',
    remove_unused_columns = False,
    label_names = ["labels"]
)

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_data,
    tokenizer=tokenizer
)

In [ ]:
trainer.train()

In [ ]:
model.save_pretrained("./tinyllama-lora-tuned-adapter-kubernetes")
tokenizer.save_pretrained("./tinyllama-lora-tuned-adapter-kubernetes")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
